# Enlace para compartir el notebook.
## Link de google drive:
### https://drive.google.com/drive/folders/14QT8fRFuvPCK4_odhFFdbsfRGmtf9dA6?usp=sharing
## Link de la presentación PDF:
### https://drive.google.com/file/d/1_syOrVD4dz2ZR1cATY-NmQnP8hBTudOO/view?usp=sharing


# Conectarse a la base de datos

In [ ]:
# import libraries
import pandas as pd
from sqlalchemy import create_engine
import os

# ── CREDENCIALES (configura tus variables de entorno antes de ejecutar) ──────
# En tu terminal ejecuta:
#   export DB_USER="tu_usuario"
#   export DB_PWD="tu_contraseña"
#   export DB_HOST="tu_host"
#
# O reemplaza os.getenv(...) por tus valores si ejecutas en un entorno privado.

db_config = {
    'user': os.getenv('DB_USER', 'tu_usuario'),
    'pwd':  os.getenv('DB_PWD',  'tu_contraseña'),
    'host': os.getenv('DB_HOST', 'tu_host'),
    'port': 5432,
    'db':   'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})
print("Conexión configurada. Asegúrate de haber definido las variables de entorno.")

In [ ]:
# La conexión se almacena en la variable engine .
# ejecuto una consulta SQL utilizando pandas:
pd.io.sql.read_sql("SELECT * FROM information_schema.tables", con = engine)

# Proyecto SQL

### Objetivo del estudio:
Desarrollar nuevas aplicaciones para los amantes de los libros, tomando información de uno de los servicios que compiten en este mercado. Contiene datos sobre libros, editoriales, autores y calificaciones de clientes y reseñas de libros. Esta información se utilizará para generar una propuesta de valor para un nuevo producto

#### Objetivos específicos:
* **Identificar tendencias de publicación:** Cuantificar el volumen de libros lanzados en el nuevo milenio.
*  **Evaluar la recepción del contenido:** Determinar el volumen de retroalimentación y la puntuación media por obra.
*  **Analizar la relevancia de las editoriales:** Filtrar publicaciones menores (folletos) para descubrir los proveedores de contenido principales.
*  **Reconocer autores de éxito comercial:** Encontrar los escritores mejor valorados basándose en un volumen de muestra estadísticamente representativo.
*  **Perfil de usuarios críticos:** Medir el nivel de interacción textual de los lectores más activos de la plataforma.




### Se ejecuta SQL con esta sentencia:
pd.io.sql.read_sql(query, con = engine)

#### Task:
1. Encontrar el número de libros publicados después del 1 de enero de 2000.
2. Encontrar el número de reseñas de usuarios y la calificación promedio para cada libro.
3. Identificar la editorial que ha publicado el mayor número de libros con más de 50 páginas (esto te ayudará a excluir folletos y publicaciones similares de tu análisis).
4. Identificar al autor que tiene la más alta calificación promedio del libro: mira solo los libros con al menos 50 calificaciones.
5. Encontrar el número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

# Desarrollo del Proyecto
## Paso 1: Estudiar las tablas (visualizar las primeras 5 filas)

In [ ]:
# hago una lista con los nombres de los DataFrames

dataframes_proyecto = {
    "Tabla: Books (Libros)": pd.io.sql.read_sql("SELECT * FROM books LIMIT 5;", con = engine),
    "Tabla: Authors (Autores)": pd.io.sql.read_sql("SELECT * FROM authors LIMIT 5;", con = engine),
    "Tabla: Publishers (Editoriales)": pd.io.sql.read_sql("SELECT * FROM publishers LIMIT 5;", con = engine),
    "Tabla: Ratings (Calificaciones)": pd.io.sql.read_sql("SELECT * FROM ratings LIMIT 5;", con = engine),
    "Tabla: Reviews (Reseñas)": pd.io.sql.read_sql("SELECT * FROM reviews LIMIT 5;", con = engine)
}

for nombre, df in dataframes_proyecto.items():
    print(f"\n{'='*40}\n{nombre}\n{'='*40}")
    
    display(df.head(5))

## Paso 2: Confirmar la calidad de los datos

In [ ]:
import pandas as pd

# Lista con los nombres de las tablas en la base de datos
tablas = ['books', 'authors', 'publishers', 'ratings', 'reviews']

print("INICIANDO AUDITORÍA DE CALIDAD DE DATOS...\n")

for tabla in tablas:
    print(f"{'='*50}\nANÁLISIS DE LA TABLA: {tabla.upper()}\n{'='*50}")
    
    # Traer la tabla completa para analizarla con Pandas
    df = pd.io.sql.read_sql(f"SELECT * FROM {tabla};", con=engine)
    
    # 1. Verificar Tipos de Datos y Valores No Nulos
    print("Estructura y tipos de datos:")
    info_df = pd.DataFrame({
        'Tipo de Dato': df.dtypes,
        'Registros No Nulos': df.count(),
        'Total Filas': len(df)
    })
    display(info_df)
    
    # 2. Contar explícitamente los Valores Nulos (NaN)
    nulos = df.isnull().sum()
    print("Valores nulos detectados por columna:")
    df_nulos = pd.DataFrame(nulos, columns=['Cantidad de Nulos'])
    display(df_nulos)
    
    # 3. Alerta visual si se encuentran nulos
    total_nulos = nulos.sum()
    if total_nulos > 0:
        print(f" ¡ATENCIÓN! Se encontraron {total_nulos} valores nulos en la tabla '{tabla}'.")
    else:
        print(f"Tabla OK '{tabla}' limpia. 0 valores nulos encontrados.")
    print("\n")


## Paso 3: Desarrollar el proyecto: Obtener los restultados de cada tarea

In [ ]:
# task 1: Número de libros publicados después del 1 de enero de 2000
query1= """
SELECT COUNT(book_id) AS total_libros_2000
FROM books
WHERE publication_date::DATE  > '2000-01-01';
"""
task1 = pd.io.sql.read_sql(query1, con = engine)
print("1. Libros publicados después del 1 de enero de 2000:")
display(task1)

In [ ]:
# task 2: Número de reseñas de usuarios y calificación promedio para cada libro
query2 = """
SELECT 
    b.book_id,
    b.title,
    COALESCE(rev.num_resenas, 0) AS num_resenas,
    COALESCE(rat.calificacion_promedio, 0) AS calificacion_promedio
FROM books b
LEFT JOIN (
    SELECT book_id, COUNT(review_id) AS num_resenas
    FROM reviews
    GROUP BY book_id
) rev ON b.book_id = rev.book_id
LEFT JOIN (
    SELECT book_id, AVG(rating) AS calificacion_promedio
    FROM ratings
    GROUP BY book_id
) rat ON b.book_id = rat.book_id;
"""
task2 = pd.io.sql.read_sql(query2, con = engine)
print("\n2. Reseñas y calificación promedio por libro (Muestra de 10):")
display(task2.head(10))

In [ ]:
# task 3: Editorial con el mayor número de libros con más de 50 páginas

query3 = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_libros
FROM books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher_id, p.publisher
ORDER BY total_libros DESC
LIMIT 1;
"""
task3 = pd.io.sql.read_sql(query3, con = engine)
print("\n3. Editorial con más libros de >50 páginas (Aliado estratégico potencial):")
display(task3)

In [ ]:
# task 4: Autor con la más alta calificación promedio (libros con >= 50 calificaciones)
query4 = """
WITH libros_populares AS (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
),
promedio_por_libro AS (
    SELECT book_id, AVG(rating) AS promedio_libro
    FROM ratings
    WHERE book_id IN (SELECT book_id FROM libros_populares)
    GROUP BY book_id
)
SELECT 
    a.author,
    AVG(pl.promedio_libro) AS promedio_autor
FROM promedio_por_libro pl
JOIN books b ON pl.book_id = b.book_id
JOIN authors a ON b.author_id = a.author_id
GROUP BY a.author_id, a.author
ORDER BY promedio_autor DESC
LIMIT 1;
"""
task4 = pd.io.sql.read_sql(query4, con = engine)
print("\n4. Autor mejor calificado con base estadística sólida (>= 50 calificaciones):")
display(task4)

In [ ]:
# task 5: Número promedio de reseñas de texto de usuarios con >50 libros calificados

query5 = """
WITH usuarios_activos AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 50
),
resenas_por_usuario AS (
    SELECT username, COUNT(review_id) AS conteo_resenas
    FROM reviews
    WHERE username IN (SELECT username FROM usuarios_activos)
    GROUP BY username
)
SELECT AVG(conteo_resenas) AS promedio_resenas_texto
FROM resenas_por_usuario;
"""
task5 = pd.io.sql.read_sql(query5, con = engine)
print("\n5. Promedio de reseñas de texto escritas por usuarios críticos:")
display(task5)

## Paso 4: Conclusiones de cada tarea

### Tarea 1: Número de libros publicados después del 1 de enero de 2000
* **Resultado:** 819 libros publicados después del 1 de enero de 2000.
* **Conclusión:** El hecho de tener 819 títulos del nuevo milenio demuestra que la plataforma cuenta con un catálogo moderno robusto, representando una oferta atractiva para el lector actual que busca tendencias contemporáneas. Al no depender de literatura clásica o de dominio público, la startup se posiciona con un inventario fresco y comercial, lo que facilita el diseño de campañas de marketing enfocadas en la actualidad del contenido.

### Tarea 2: Número de reseñas de usuarios y calificación promedio para cada libro
* **Resultado:** Muestra con un promedio de 2 a 5 reseñas por libro y calificaciones altas entre 4.0 y 4.50.
* **Conclusión:** Los datos revelan un patrón de alta calidad en las lecturas, donde títulos icónicos como "A Tree Grows in Brooklyn" o "American Gods" logran el mayor engagement (5 reseñas). Esto indica que los usuarios no solo consumen el contenido, sino que están activamente motivados a evaluarlo. Para el nuevo producto, esta densidad de interacciones permite crear un sistema de recomendación confiable basado en el volumen de críticas, evitando el riesgo de recomendar libros con opiniones aisladas.

### Tarea 3: Editorial con el mayor número de libros con más de 50 páginas
* **Resultado:** Penguin Books lidera con 42 libros de más de 50 páginas.
* **Conclusión:** Penguin Books se consolida como el jugador clave y el proveedor de contenido más importante del ecosistema actual. Un catálogo de 42 obras extensas posiciona a esta editorial como el aliado B2B prioritario para la startup. Cerrar un acuerdo de distribución o exclusividad con ellos garantiza asegurar casi el 5% de la oferta total relevante de golpe, optimizando los esfuerzos comerciales y los costos de adquisición de licencias.

### Tarea 4: Autor con la más alta calificación promedio (libros con >= 50 calificaciones)
* **Resultado:** J.K. Rowling / Mary GrandPré con una calificación promedio de 4.28 (en libros con mayor e igual a 50 calificaciones).
* **Conclusión:** Mantener una puntuación de 4.28 bajo la presión de una muestra estadísticamente grande (más de 50 calificaciones por libro) confirma un éxito comercial y de crítica implacable. Esta métrica elimina el sesgo de autores con calificaciones perfectas pero pocos lectores. El nuevo producto debe colocar a este perfil de autores como el pilar de retención principal en la pantalla de inicio; promocionarlos garantiza una alta probabilidad de satisfacción en la primera experiencia de lectura del usuario, reduciendo el abandono de la app.

### Tarea 5: Número promedio de reseñas de texto de usuarios con >50 libros calificados
* **Resultado:** Un promedio de 24.33 reseñas de texto por cada usuario crítico.
* **Conclusión:** Este es uno de los hallazgos más potentes para el negocio. Los usuarios de alto consumo (más de 50 libros calificados) no solo puntúan, sino que se toman el tiempo de redactar más de 24 reseñas escritas detalladas en promedio. Este grupo representa el motor de contenido orgánico gratuito (UGC) de la plataforma. La startup debe proteger a este nicho mediante un programa de lealtad o gamificación ("Crítico Élite"), ya que sus más de 24 reseñas por cabeza son el gancho social que convence a los usuarios nuevos de quedarse en la aplicación.